In [ ]:
import subprocess
import time
import re

# 1. Install dependencies & Ollama + Cloudflared
!apt-get update -qq && apt-get install -y -qq pciutils lshw zstd
!curl -fsSL https://ollama.com/install.sh | sh
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

# 2. Start Ollama server in the background
import os
os.environ["OLLAMA_HOST"] = "0.0.0.0:11434"
os.environ["OLLAMA_ORIGINS"] = "*"
os.environ["OLLAMA_KEEP_ALIVE"] = "-1"
os.environ["OLLAMA_FLASH_ATTENTION"] = "1"

ollama_proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(3)

# 3. Pull whatever model onto the whatever gpu you want
print("Pulling qwen2.5-coder:7b...")
!ollama pull qwen2.5-coder:7b
print("Pulling qwen2.5-coder:14b...")
!ollama pull qwen2.5-coder:14b
print("Pulling qwen2.5-coder:32b...")
!ollama pull qwen2.5-coder:32b

# 4. Start Cloudflare Tunnel (rewriting Host header so Ollama accepts remote requests)
tunnel_proc = subprocess.Popen(
    [
        "cloudflared", "tunnel",
        "--url", "http://localhost:11434",
        "--http-host-header", "localhost:11434"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

public_url = None
for line in tunnel_proc.stdout:
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break

print("\n" + "="*65)
print(f"COLAB GPU SERVER READY!")
print(f"Paste this into configs/experiment.yaml under base_url:")
print(f'  base_url: "{public_url}/v1"')
print("="*65 + "\n")

# Keep the Colab cell alive while you run tests from PyCharm
try:
    while True:
        time.sleep(60)
except KeyboardInterrupt:
    print("Tunnel stopped.")